# Test `mutate_conv3x3`
Tests that `mutate_conv3x3` correctly re-orders the output channels (and their corresponding BN stats) from most-to-least important based on L2-norm of each filter.

In [1]:
import sys
sys.path.insert(0, r"D:\Sutanu_WorkSpace\GA-NAS\GA-NAS")

import torch
from GA.Blocks import *
from GA.Mutation import *

print("Imports OK")

Imports OK


In [2]:
# ── Create a Conv3x3 block with random weights ────────────────────────────────
torch.manual_seed(0)

IN_CH, OUT_CH, STRIDE = 8, 16, 1
block = BasicBlock(IN_CH, OUT_CH, STRIDE)

# Compute per-output-channel importance BEFORE mutation
importances_before = get_input_channel_importance(block.conv1.weight)

print("=== BEFORE mutation ===")
print(f"Conv weight shape : {block.conv1.weight.shape}")   # [OUT_CH, IN_CH, 3, 3]
print(f"BN weight shape   : {block.bn1.weight.shape}")     # [OUT_CH]
print(f"\nChannel importances (output dim 0):")
for i, v in enumerate(importances_before):
    print(f"  ch {i:>2}: {v.item():.4f}")

=== BEFORE mutation ===
Conv weight shape : torch.Size([16, 8, 3, 3])
BN weight shape   : torch.Size([16])

Channel importances (output dim 0):
  ch  0: 0.5667
  ch  1: 0.5856
  ch  2: 0.5627
  ch  3: 0.5776
  ch  4: 0.5328
  ch  5: 0.6286
  ch  6: 0.6337
  ch  7: 0.5614
  ch  8: 0.6242
  ch  9: 0.5808
  ch 10: 0.6030
  ch 11: 0.5002
  ch 12: 0.6379
  ch 13: 0.5813
  ch 14: 0.5803
  ch 15: 0.5840


In [3]:
for name, param in block.named_parameters():
    print(f"Project {name} shape: {param.shape}")

Project conv1.weight shape: torch.Size([16, 8, 3, 3])
Project bn1.weight shape: torch.Size([16])
Project bn1.bias shape: torch.Size([16])
Project conv2.weight shape: torch.Size([16, 16, 3, 3])
Project bn2.weight shape: torch.Size([16])
Project bn2.bias shape: torch.Size([16])
Project downsample.0.weight shape: torch.Size([16, 8, 1, 1])
Project downsample.1.weight shape: torch.Size([16])
Project downsample.1.bias shape: torch.Size([16])


In [4]:
for tensor_name in ['weight', 'bias', 'running_mean', 'running_var']:
    tensor_to_apply = getattr(block.bn1, tensor_name)
    print(f"\nBN {tensor_name} BEFORE mutation:")
    for i in range(OUT_CH):
        print(f"  ch {i:>2}: {tensor_to_apply[i].item():.4f}")



BN weight BEFORE mutation:
  ch  0: 1.0000
  ch  1: 1.0000
  ch  2: 1.0000
  ch  3: 1.0000
  ch  4: 1.0000
  ch  5: 1.0000
  ch  6: 1.0000
  ch  7: 1.0000
  ch  8: 1.0000
  ch  9: 1.0000
  ch 10: 1.0000
  ch 11: 1.0000
  ch 12: 1.0000
  ch 13: 1.0000
  ch 14: 1.0000
  ch 15: 1.0000

BN bias BEFORE mutation:
  ch  0: 0.0000
  ch  1: 0.0000
  ch  2: 0.0000
  ch  3: 0.0000
  ch  4: 0.0000
  ch  5: 0.0000
  ch  6: 0.0000
  ch  7: 0.0000
  ch  8: 0.0000
  ch  9: 0.0000
  ch 10: 0.0000
  ch 11: 0.0000
  ch 12: 0.0000
  ch 13: 0.0000
  ch 14: 0.0000
  ch 15: 0.0000

BN running_mean BEFORE mutation:
  ch  0: 0.0000
  ch  1: 0.0000
  ch  2: 0.0000
  ch  3: 0.0000
  ch  4: 0.0000
  ch  5: 0.0000
  ch  6: 0.0000
  ch  7: 0.0000
  ch  8: 0.0000
  ch  9: 0.0000
  ch 10: 0.0000
  ch 11: 0.0000
  ch 12: 0.0000
  ch 13: 0.0000
  ch 14: 0.0000
  ch 15: 0.0000

BN running_var BEFORE mutation:
  ch  0: 1.0000
  ch  1: 1.0000
  ch  2: 1.0000
  ch  3: 1.0000
  ch  4: 1.0000
  ch  5: 1.0000
  ch  6: 1.0000

In [5]:
# ── Run mutate_conv3x3 ────────────────────────────────────────────────────────
mutate_basic_block(block,new_out_channels=80)

importances_after = get_input_channel_importance(block.conv1.weight)

print("=== AFTER mutation ===")
print(f"\nChannel importances (should be descending):")
for i, v in enumerate(importances_after):
    
    print(f"  ch {i:>2}: {v.item():.4f}")

# Verify strict descending order
is_sorted = all(
    importances_after[i] >= importances_after[i + 1]
    for i in range(len(importances_after) - 1)
)
print(f"\n✓ Output channels sorted by importance (descending): {is_sorted}")

=== AFTER mutation ===

Channel importances (should be descending):
  ch  0: 0.6379
  ch  1: 0.6337
  ch  2: 0.6286
  ch  3: 0.6242
  ch  4: 0.6030
  ch  5: 0.5856
  ch  6: 0.5840
  ch  7: 0.5813
  ch  8: 0.5808
  ch  9: 0.5803
  ch 10: 0.5776
  ch 11: 0.5667
  ch 12: 0.5627
  ch 13: 0.5614
  ch 14: 0.5328
  ch 15: 0.5002
  ch 16: 0.0000
  ch 17: 0.0000
  ch 18: 0.0000
  ch 19: 0.0000
  ch 20: 0.0000
  ch 21: 0.0000
  ch 22: 0.0000
  ch 23: 0.0000
  ch 24: 0.0000
  ch 25: 0.0000
  ch 26: 0.0000
  ch 27: 0.0000
  ch 28: 0.0000
  ch 29: 0.0000
  ch 30: 0.0000
  ch 31: 0.0000
  ch 32: 0.0000
  ch 33: 0.0000
  ch 34: 0.0000
  ch 35: 0.0000
  ch 36: 0.0000
  ch 37: 0.0000
  ch 38: 0.0000
  ch 39: 0.0000
  ch 40: 0.0000
  ch 41: 0.0000
  ch 42: 0.0000
  ch 43: 0.0000
  ch 44: 0.0000
  ch 45: 0.0000
  ch 46: 0.0000
  ch 47: 0.0000
  ch 48: 0.0000
  ch 49: 0.0000
  ch 50: 0.0000
  ch 51: 0.0000
  ch 52: 0.0000
  ch 53: 0.0000
  ch 54: 0.0000
  ch 55: 0.0000
  ch 56: 0.0000
  ch 57: 0.0000
  ch

In [6]:
for name, param in block.named_parameters():
    print(f"Project {name} shape: {param.shape}")

Project conv1.weight shape: torch.Size([80, 8, 3, 3])
Project bn1.weight shape: torch.Size([80])
Project bn1.bias shape: torch.Size([80])
Project conv2.weight shape: torch.Size([80, 80, 3, 3])
Project bn2.weight shape: torch.Size([80])
Project bn2.bias shape: torch.Size([80])
Project downsample.0.weight shape: torch.Size([80, 8, 1, 1])
Project downsample.1.weight shape: torch.Size([80])
Project downsample.1.bias shape: torch.Size([80])


In [7]:
# ── Verify BN tensors are reordered consistently ─────────────────────────────
# After mutation, the BN stats must follow the same permutation as conv weights.
# We check by re-computing the sort index from the mutated conv and confirming
# the BN weight/bias/running_mean/running_var are already in that order.
sort_idx_after = sort_channels_by_importance(block.conv1.weight)

print("Sort index after mutation (should be 0,1,2,...  i.e. already sorted):")
print(sort_idx_after.tolist())

print("\nBN weight (first 8)   :", block.bn1.weight.detach()[:].tolist())
print("BN bias   (first 8)   :", block.bn1.bias.detach()[:].tolist())
print("BN run_mean (first 8) :", block.bn1.running_mean[:].tolist())
print("BN run_var  (first 8) :", block.bn1.running_var[:].tolist())

# Forward-pass sanity check – should not raise
dummy = torch.randn(2, IN_CH, 32, 32)
block.eval()
out = block(dummy)
print(f"\nForward pass OK  →  output shape: {out.shape}")

Sort index after mutation (should be 0,1,2,...  i.e. already sorted):
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79]

BN weight (first 8)   : [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
BN bias   (first 8)   : [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0